## Generate the complete USGS point locations table using USGS dataretrieval

### 1. Get locations with continuous timeseries (waterdata.get_time_series_metadata)
- Primary codes:
    - 00060	Discharge, cubic feet per second
    - 00065	Gage height, feet
- Less common codes included to catch edge cases:
    - 00061	Discharge, instantaneous  
    - 72137	Discharge, tidally filtered, cubic feet per second  
    - 99133	Computed discharge
    - 30208	Stream stage (alternate coding)
    - 63160	Stream water level elevation above NAVD 1988, in feet
    - 00062	Elevation of reservoir water surface above datum, feet
    - 62614	Lake or reservoir water surface elevation above NGVD 1929, feet
    - 62615	Lake or reservoir water surface elevation above NAVD 1988, feet
- *Note 63 locations returned with the above param codes that were found to be groundwater sites, these were assumed to be incorrectly coded and were excluded.   
- *time series with begin/end dates = NaT did not represent real data and were removed   
- *time series with dates defined but statistic_id=None were manually checked and corrected (statistic_id=00011 in most cases, notes below)   

### 2. Get station name and drainage area (if available) (waterdata.get_monitoring_locations)
- Using the get_monitoring_locations method is necessary to get names and areas
- Included site_type_codes that reported any of the parameter codes above (surface water)
    - stream_codes: 'ST','ST-TS','ST-CA','ST-DCH','SP'
    - lake_codes: 'LK','LA','LA-EX','LA-OU','LA-PLY','LA-SH','LA-SNK','LA-SR','LA-VOL'
    - infrastructure_codes: 'FA-CI','FA-CS','FA-DV','FA-FON','FA-HP','FA-OF','FA-STS','FA-TEP','FA-WDS','FA-WIW','FA-WTP','FA-WWD','FA-WWTP','SB-TSM'
    - coastal_wetland_codes: 'ES','OC-CO','OC','WE'
- site_type_codes that contain any surface discharge or water level time series were identified through trial and error

### 3. Update attribute tables
- From location and time series metadata:
    - state_name
    - state_abbrev
    - county_name
    - drainage_area
    - site_type_code
    - is_active
    - first_year
    - last_year
    - max_record
    - num_timeseries 
    - has_inst_discharge
    - has_mean_daily_discharge
    - has_max_daily_discharge
    - has_inst_stage
    - has_mean_daily_stage
    - has_max_daily_stage
- Spatial overlay
    - rfc
    - wfo
    - nws_region
    - huc12
    - EPA level 1 and 2 ecoregions
- Calculated from point coordinates
    - iana_timezone ---requires: timezonefinder

In [ ]:
!pip install timezonefinder

In [ ]:
import pandas as pd
import geopandas as gpd
from dataretrieval import waterdata
from pathlib import Path
from datetime import datetime
from timezonefinder import TimezoneFinder

import teehr

# set USGS token - remove if not required (add your token here)
import os
os.environ["API_USGS_PAT"] = "yKX8FeeBd4RacU01Pwe3Mbw7GbqGz0UVEeboL25E"

### Fetch locations and time series metadata

In [ ]:
# include all states and territories
states = [
    ("Alabama","AL"), ("Alaska","AK"), ("Arizona","AZ"), ("Arkansas","AR"),
    ("California","CA"), ("Colorado","CO"), ("Connecticut","CT"), ("Delaware","DE"),
    ("Florida","FL"), ("Georgia","GA"), ("Hawaii","HI"), ("Idaho","ID"),
    ("Illinois","IL"), ("Indiana","IN"), ("Iowa","IA"), ("Kansas","KS"),
    ("Kentucky","KY"), ("Louisiana","LA"), ("Maine","ME"), ("Maryland","MD"),
    ("Massachusetts","MA"), ("Michigan","MI"), ("Minnesota","MN"), ("Mississippi","MS"),
    ("Missouri","MO"), ("Montana","MT"), ("Nebraska","NE"), ("Nevada","NV"),
    ("New Hampshire","NH"), ("New Jersey","NJ"), ("New Mexico","NM"), ("New York","NY"),
    ("North Carolina","NC"), ("North Dakota","ND"), ("Ohio","OH"), ("Oklahoma","OK"),
    ("Oregon","OR"), ("Pennsylvania","PA"), ("Rhode Island","RI"), ("South Carolina","SC"),
    ("South Dakota","SD"), ("Tennessee","TN"), ("Texas","TX"), ("Utah","UT"),
    ("Vermont","VT"), ("Virginia","VA"), ("Washington","WA"), ("West Virginia","WV"),
    ("Wisconsin","WI"), ("Wyoming","WY"), ("Puerto Rico","PR"), ("Virgin Islands", "USVI"),
    ("Guam", "GM"), ("American Samoa", "AS"), ("Northern Mariana Islands", "CNMI")
]

# for monitoring location queries - include the following site_codes (determined through trial and error)
stream_codes = ['ST','ST-TS','ST-CA','ST-DCH','SP']
lake_codes = ['LK','LA','LA-EX','LA-OU','LA-PLY','LA-SH','LA-SNK','LA-SR','LA-VOL']
inf_codes = ['FA-CI','FA-CS','FA-DV','FA-FON','FA-HP','FA-OF','FA-STS','FA-TEP','FA-WDS','FA-WIW','FA-WTP','FA-WWD','FA-WWTP','SB-TSM']
coastal_wetland_codes = ['ES','OC-CO','OC','WE']
all_codes = stream_codes + lake_codes + inf_codes + coastal_wetland_codes

# initialize lists
all_ts_list = []
all_loc_list = []

# Get the current date and time
current_datetime = datetime.now()
fetch_date = current_datetime.strftime("%m-%d-%Y")
fetch_date_in_path = fetch_date.replace('-','')

# query by state
for state_tuple in states:
    state_name = state_tuple[0]
    state_abbrev = state_tuple[1]
    print(f"Processing {state_name}")

    # Get continuous timeseries metadata -
    #      the results include metadata for locations that contain
    #      continuous (instantaneous or daily) data.
    df_ts, metadata = waterdata.get_time_series_metadata(
        state_name=state_name,
        parameter_code=['00054','00060','00061','00062','00065','30207','30208','62614','62615','63160','72137','99133'],
        properties=[
            'monitoring_location_id',
            'geometry',
            'statistic_id',
            'state_name',
            'begin',
            'end',
            'parameter_code',
        ]
    )
    df_ts = df_ts.rename(columns = {'monitoring_location_id':'id'})

    # Get monitoring location information
    df_locs, metadata = waterdata.get_monitoring_locations(
        state_name=state_name,
        site_type_code=all_codes,
        properties=[
            'monitoring_location_id',
            'monitoring_location_name',
            'state_name',
            'county_name',
            'drainage_area',
            'geometry',
            'site_type_code',
        ]
    )
    df_locs['state'] = state_abbrev
    df_locs = df_locs.rename(columns = {'monitoring_location_id':'id', 'monitoring_location_name':'name'})

    # add the current state results to the list
    all_ts_list.append(df_ts)
    all_loc_list.append(df_locs)

print("concatenating")
all_ts = pd.concat(all_ts_list, ignore_index=True)
all_locs = pd.concat(all_loc_list, ignore_index=True)

### Filter / fix issues

In [ ]:
# remove rows with missing dates - spot checks confirmed these do not represent any data
all_ts_dropna_dates = all_ts[~all_ts['begin'].isna()]

# drop 3 locations with missing geometry - coordinates not found in all searches
all_ts_dropna_geom = all_ts_dropna_dates[~all_ts_dropna_dates['geometry'].isna()].sort_values('id')

# locations where statistic_id was None but dates existed (suspected errors):
statistic_id_none = all_ts_dropna_geom[all_ts_dropna_geom['statistic_id'].isna()].sort_values('id')
print(f"{len(statistic_id_none)} locations where statistic_id was None but dates existed (suspected errors):")
display(statistic_id_none)

# drop 1 time series with statistic_id=None at location USGS-08188500
# (appears to be incorrectly coded nitrate data as discharge)
all_ts_drop_loc = all_ts_dropna_geom[
    ~((all_ts_dropna_geom['statistic_id'].isna()) &
    (all_ts_dropna_geom['id'] == 'USGS-08188500'))
    ]
all_ts_drop_loc

# set statistic_id = '00011' in all others where it was None - manual check at 14 locations
all_ts_drop_loc.loc[all_ts_drop_loc['statistic_id'].isna(), 'statistic_id'] = "00011"

####  14 timeseries manually checked:
USGS-01208760 - has inst stage and discharge, (no daily)                 - None stat w dates should be 00011   
USGS-02246518 - has inst stage and discharge, daily stage and discharge  - None stat w dates should be 00011   
USGS-06129000 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-09423560 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-09429070 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-09502830 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-04121500 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-04122500 - has inst stage and discharge, daily discharge,           - None stat w dates should be 00011   
USGS-01411175 - has inst discharge                                       - None stat w dates should be 00011   
USGS-02492200 - has inst discharge                                       - None stat w dates should be 00011   
USGS-07380125 - has inst discharge                                       - None stat w dates should be 00011   
USGS-03291585 - has inst and daily stage                                 - None stat w dates should be 00011   
USGS-05414213 - has inst and daily discharge                             - None stat w dates should be 00011   
USGS-08188500 - has inst and daily discharge, daily stage and discharge  - None stat w dates should be dropped (nitrate, incorrect param code)

In [ ]:
# A small number of non-USGS locations are included in results (USFS, USBWI (boundary water), state networks).
# These are excluded from the TEEHR list for now.
all_ts_usgs = all_ts_drop_loc[all_ts_drop_loc['id'].str.contains('USGS-')].sort_values('id').reset_index().drop('index', axis=1)

# add begin/end years (for easy reference) and n_years of record
all_ts_usgs['begin_year'] = all_ts_usgs['begin'].dt.year
all_ts_usgs['end_year'] = all_ts_usgs['end'].dt.year
all_ts_usgs['n_years'] = ((all_ts_usgs['end'] - all_ts_usgs['begin']).dt.days / 365.25).round(1)

# get unique locations within the time series list
unique_locations = all_ts_usgs[['id','geometry','state_name']].drop_duplicates('id').reset_index().drop('index', axis=1)

# drop 63 locations that are not found by the monitoring locations query -
# these are groundwater sites incorrectly coded with discharge or stage param codes
final_locations = unique_locations[unique_locations['id'].isin(all_locs['id'])]

# get location meta data to add to the locations df
all_locs_meta = all_locs[['id','name','drainage_area','state','site_type_code','county_name']]
final_locations = final_locations.merge(all_locs_meta, how='left', on='id')
final_locations.set_crs("EPSG:4326", inplace=True)
properties_dict = {
    "source" : "USGS",
    "date_retrieved" : fetch_date,
    "method" : "https://github.com/DOI-USGS/dataretrieval-python",
}
final_locations['properties'] = [properties_dict] * len(final_locations)

# teehr prefixes are lower case
all_ts_usgs['id'] = all_ts_usgs['id'].str.replace('USGS','usgs')
final_locations['id'] = final_locations['id'].str.replace('USGS','usgs')

# Select columns needed for TEEHR and add properties column
locations_teehr = final_locations[['id','name','geometry','properties']].copy()

## TEEHR locations table to add to the data warehouse:

In [ ]:
locations_teehr

In [ ]:
import geopandas as gpd

locations_teehr = gpd.read_parquet("/data/common/geometry/updated_locations/teehr_locations.parquet")
locations_teehr

In [ ]:
locations_teehr.properties[0]

In [ ]:
from teehr.evaluation.evaluation import create_spark_session

spark = create_spark_session(
    aws_profile="admin-user"
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
%%time
ev.locations.load_dataframe(
    df=locations_teehr,
    write_mode="upsert"
)

In [ ]:
ev.locations.to_sdf().count()

In [ ]:
# # write file if desired
# locations_teehr.to_parquet(f'/data/common/geometry/usgs_point_geometry_{fetch_date_in_path}.parquet')

## Create attributes
- From location and time series metadata:
    - state_name
    - state_abbrev
    - county_name
    - drainage_area
    - site_type_code
    - is_active
    - first_year
    - last_year
    - max_record
    - num_timeseries 
    - has_inst_discharge
    - has_mean_daily_discharge
    - has_max_daily_discharge
    - has_inst_stage
    - has_mean_daily_stage
    - has_max_daily_stage
- Spatial overlay
    - rfc
    - wfo
    - nws_region
    - huc12
    - EPA level 1 and 2 ecoregions
- Calculated from point coordinates
    - iana_timezone ---requires: timezonefinder


In [ ]:
attribute_dict = {}

#### Monitoring location metadata
print('processing location metadata...')

def attribute_from_location_metadata(df, attribute):
    attribute_df = df[['id',attribute]].rename(columns = {attribute:'attribute_value','id':'location_id'})
    attribute_df['attribute_name'] = attribute
    properties_dict = {
        "source" : "USGS",
        "date_retrieved" : fetch_date,
        "method" : "https://github.com/DOI-USGS/dataretrieval-python",
    }
    attribute_df['properties'] = [properties_dict] * len(final_locations)
    return attribute_df

for attribute in ['state_name','state','county_name','site_type_code','drainage_area']:
    attribute_dict[attribute] = attribute_from_location_metadata(final_locations, attribute)

# convert drainage area units from mi2 to km2
attribute_dict['drainage_area']['attribute_value'] = (attribute_dict['drainage_area']['attribute_value'] * 2.5899).round(2)
attribute_dict['drainage_area']['attribute_unit'] = 'km^2'


#### time series metadata
print('processing time series metadata...')

def attribute_from_ts_meta(loc_df, ts_df, attribute, stat='00011', param_list=[]):
    attribute_df = loc_df[['id']].copy().rename(columns={'id':'location_id'})
    attribute_df['attribute_value'] = False
    attribute_df['attribute_name'] = attribute
    properties_dict = {
        "source" : "USGS",
        "date_retrieved" : fetch_date,
        "method" : "https://github.com/DOI-USGS/dataretrieval-python",
    }
    attribute_df['properties'] = [properties_dict] * len(final_locations)
    if param_list:
        location_trues = ts_df[
            (ts_df['statistic_id'] == stat) &
            (ts_df['parameter_code'].isin(param_list))
            ].drop_duplicates('id')
    else:
        # treated as active if reported in the the past week
        location_trues = ts_df[
            ts_df['end'] >= pd.Timestamp.now() - pd.Timedelta(days=7)
            ].drop_duplicates('id')
    attribute_df.loc[attribute_df['location_id'].isin(location_trues['id']), 'attribute_value'] = True
    return attribute_df

attribute_dict['has_inst_discharge'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_inst_discharge', '00011', ['00060','00061','72137'])
attribute_dict['has_mean_daily_discharge'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_mean_daily_discharge', '00003', ['00060','00061','72137'])
attribute_dict['has_max_daily_discharge'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_max_daily_discharge', '00001', ['00060','00061','72137'])
attribute_dict['has_inst_stage'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_inst_stage', '00011', ['00065','30208','63160'])
attribute_dict['has_mean_daily_stage'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_mean_daily_stage', '00003', ['00065','30208','63160'])
attribute_dict['has_max_daily_stage'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'has_max_daily_stage', '00001', ['00065','30208','63160'])
attribute_dict['is_active'] = attribute_from_ts_meta(final_locations, all_ts_usgs, 'is_active')

#### time series summaries
print('processing time series summaries..')

# aggregate across time series
all_ts_agg = all_ts_usgs.groupby('id').agg(
    max_record = ('n_years', 'max'),
    first_year = ('begin_year','min'),
    last_year = ('end_year','max'),
    num_timeseries = ('statistic_id','count')
).reset_index()

def attribute_from_ts_sum(loc_df, sum_df, attribute):
    attribute_df = loc_df[['id']].merge(sum_df[['id',attribute]], how='left', on='id')
    attribute_df = attribute_df.rename(columns={'id':'location_id',attribute:'attribute_value'})
    attribute_df['attribute_name'] = attribute
    properties_dict = {
        "source" : "USGS",
        "date_retrieved" : fetch_date,
        "method" : "https://github.com/DOI-USGS/dataretrieval-python",
    }
    attribute_df['properties'] = [properties_dict] * len(final_locations)
    return attribute_df

for attribute in ['first_year','last_year','num_timeseries','max_record']:
    attribute_dict[attribute] = attribute_from_ts_sum(final_locations, all_ts_agg, attribute)

# add unit record length
attribute_dict['max_record']['attribute_unit'] = 'years'

for key in attribute_dict.keys():
    display(attribute_dict[key])

In [ ]:
#### useful spatial metadata
print('processing spatial overlays...')

def attribute_from_spatial_overlay(points, attribute, polygon_s3_url, poly_id):

    polygons = gpd.read_parquet(polygon_s3_url)

    # get source from polygons if exists
    try:
        source = polygons['properties'][0]['source']
    except:
        source = "unknown"
    polygons = polygons[[poly_id,'geometry']].rename(columns={poly_id:'id'}).copy()
    points = points.to_crs(polygons.crs)
    joined = gpd.sjoin(points, polygons, how="left", predicate="within")
    joined = joined.drop('index_right', axis=1)
    attribute_df = joined[['id_left','id_right']].copy()
    attribute_df = attribute_df.rename(columns = {'id_left':'location_id', 'id_right':'attribute_value'})
    attribute_df = attribute_df.groupby('location_id').first().reset_index()
    attribute_df['attribute_name'] = attribute
    properties_dict = {
        "source" : source,
        "date_calculated" : fetch_date,
        "method" : "geopandas.sjoin"
    }
    attribute_df['properties'] = [properties_dict] * len(final_locations)

    return attribute_df

# geometry polygons needed for attribute overlays
files = {
    "rfc":"rfc_geometry.all.parquet",
    "huc12":"huc12_geometry.all.parquet",
    "wfo":"wfo_geometry.all.parquet",
    "nws_region":"nws_region_geometry.all.parquet",
    "epa_ecoregion_l1":"epa_ecoregion_l1_geometry.all.parquet",
    "epa_ecoregion_l2":"epa_ecoregion_l2_geometry.all.parquet"
}
for key in files.keys():
    url = f"s3://ciroh-rti-public-data/teehr-data-warehouse/common/geometry/{files[key]}"
    attribute_dict[key] = attribute_from_spatial_overlay(final_locations, key, url, 'id')
    print(key)

print("Complete")

In [ ]:
# IANA timezone, determined by lat-lon coords
print('processing time zones...')

from timezonefinder import TimezoneFinder
tf = TimezoneFinder()
locations_add_tz = locations_teehr.copy()
locations_add_tz['timezone'] = None
for i, row in locations_add_tz.iterrows():
    lon = row['geometry'].x
    lat = row['geometry'].y
    timezone_str = tf.certain_timezone_at(lat=lat, lng=lon)
    locations_add_tz.loc[i, 'timezone'] = timezone_str
attribute_df = locations_add_tz[['id','timezone']].copy()
attribute_df['attribute_name'] = 'iana_timezone'
attribute_df = attribute_df.rename(columns = {'id':'location_id', 'timezone':'attribute_value'})
properties_dict = {
    "source" : "calculated",
    "date_calculated" : fetch_date,
    "method" : "timezonefinder"
}
attribute_df['properties'] = [properties_dict] * len(final_locations)
attribute_dict['iana_timezone'] = attribute_df
print('attributes complete')

## TEEHR location attributes to add to the data warehouse:

In [ ]:
for key in files.keys():
    display(attribute_dict[key])
display(attribute_dict['iana_timezone'])

## write files to /data/common/attributes if desired

In [ ]:
ATTR_DIR = Path("/data/common/attributes/updated_attrs/")

In [ ]:
attribute_dict[key].rename(columns={"attribute_value": "value"})

In [ ]:
for key in attribute_dict.keys():
    filepath = Path(ATTR_DIR, f"usgs_point_attr_{key}.all.parquet")
    df = attribute_dict[key]
    df.rename(columns={"attribute_value": "value"}, inplace=True)
    df.dropna(subset=["value"], inplace=True)
    df.to_parquet(filepath)
    print(filepath)

In [ ]:
pd.read_parquet("/data/common/attributes/updated_attrs/usgs_point_attr_is_active.all.parquet")

In [ ]:
%%time
ev.location_attributes.load_parquet(
    in_path=ATTR_DIR,
    write_mode="upsert",
    update_attrs_table=True
)

In [ ]:
ev.location_attributes.to_sdf().select("attribute_name").distinct().count()

In [ ]:
ev.attributes.to_sdf().show(n=100)